# 步骤 06 · 共振带凭什么选 —— 这一步才是你自己的工作

**这一节的产出：图 6a（频带地图）+ 图 6b（两种选带准则的对比）+ 图 6c（跨故障尺寸的稳健性）+ 一条可辩护的诊断规则。**

## 前五步任何人照做都能做出来

步骤 01–05 是一条已知的路：波形 → 统计量 → FFT 失败 → 时频图 → 包络谱。每一步都有标准答案，照着做就行。

**这一步没有标准答案。**

我在步骤 05 用的是 `BAND = (2000, 4000)`。这个数字的来源是：我看了一眼图 3a，觉得能量鼓包大概在那儿，手一划定的。

**"我看着像"不是理由。** 这一节要把它变成理由。

## 你已经发现的三个事实

练习里你自己扫出来的：

| | 最佳频带 | 峰背比 |
|---|---|---|
| 外圈 | 2500–3500 | 682 |
| 内圈 | 1000–2000 | 373 |
| 健康 | 500–1500 | **7.9** |

**外圈和内圈的最佳频带不一样。** 那"该选哪个"就不是一个可以蒙混过去的问题了。

而健康记录能刷到 7.9 —— 这说明这个指标本身会给一个好轴承打出不低的分。

---
## 1. 先看清一个陷阱：那个选法是循环论证

我们一直在用"峰背比最高的频带"来挑。问题是：

> **要算峰背比，你得先知道去哪个频率量峰。**

外圈记录我们量 BPFO，内圈量 BPFI —— **因为我们已经知道答案了。**

现实中你面对一台机器，不知道它哪儿坏了。**那正是你要诊断的东西。** 用答案去挑参数，再用挑出的参数去"得出"答案，这是自己骗自己。

### 但也别矫枉过正：有些东西你是真的知道

停下来分清楚，哪些是诊断前就掌握的：

| 知道 | 为什么 |
|---|---|
| 轴承型号 SKF 6205 | 拆开看、查设备档案 |
| 它的四个频率倍数（3.5848 等） | 厂家给的几何参数 |
| 转速 1772 rpm | 转速表直接读 |
| **所以 BPFO/BPFI/BSF 三个候选频率的具体数值** | 上面两条算出来 |
| 哪一个真的坏了 | **不知道 ← 这才是要诊断的** |

**这个区分是整节的关键。**

不知道"哪个坏了"，不等于不知道"该去哪几个频率看"。三个候选频率是几何决定的，诊断之前就摆在那儿。

**所以合法的做法是**：对每个候选频带，把三个候选频率**都**评一遍分，取最高的那个作为这个频带的得分。选出得分最高的频带，同时也就选出了它支持哪个诊断。

**全程没有用到"答案"。**

---
## 2. 打分用什么 —— 谐波梳

步骤 05 里我们看到：外圈故障在包络谱上有 1×、2×、3×、4×、5× BPFO **一整列**峰。

**这比单根峰有说服力得多。** 一根孤零零的峰可能是机器谱线、可能是共振、可能是运气；**一列整数倍的峰只能是周期性冲击。**

所以打分不看单根峰，看**一整梳**：

$$\text{comb} = \sqrt[4]{r_1 \cdot r_2 \cdot r_3 \cdot r_4}$$

其中 $r_k$ 是 $k \times f_0$ 处的峰背比。

**为什么用几何平均（连乘开方）而不是算术平均？**

因为几何平均**要求每一根都在**。任何一根塌了（比如 $r_3 = 1$），整个乘积就被拖下去。算术平均则允许一根特别高的把其余的扛起来 —— 而那正是我们要排除的情况（单根高峰 = 可能是噪声）。

> 这是个很通用的技巧：**当你要求"所有条件都满足"时用几何平均，"任一条件满足即可"时用算术平均。**

---
## 3. 准备

这一节开始，稳定下来的函数搬进了 `src/envelope.py`。

理由是我们在步骤 04、05 和这里**第三次**写同一段包络代码了 —— 按之前说过的规矩，复制第三次就该封装。顺便，GitHub 上别人点进来看到的是一个有 `src/` 模块的正经工程，而不是一堆各自为战的笔记本。

In [1]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import kurtosis

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import cwru_io
import envelope as ev          # <- 新模块

DATA = ROOT / "data"
FIGURES = ROOT / "figures"
FS = cwru_io.FS

normal = cwru_io.load_baseline(DATA / "normal_1hp_98.mat",    name="Healthy")
outer  = cwru_io.load(DATA / "OR007at6_1hp_131.mat", name="Outer race")
inner  = cwru_io.load(DATA / "IR007_1hp_106.mat",    name="Inner race")
ball   = cwru_io.load(DATA / "B007_1hp_119.mat",     name="Ball")

signals = [normal, outer, inner, ball]
N = min(len(s.x) for s in signals)
TRUTH = {"Healthy": None, "Outer race": "BPFO",
         "Inner race": "BPFI", "Ball": "BSF"}
COLORS = {"Healthy": "#256049", "Outer race": "#8E3320",
          "Inner race": "#1B4F8F", "Ball": "#9A5F0A"}

GRID = ev.band_grid()          # 3 种带宽 x 若干中心
print(f"候选频带 {len(GRID)} 个，带宽 500 / 1000 / 2000 Hz，中心每 250 Hz 一档")
print(f"例如: {GRID[0]}  {GRID[len(GRID)//2]}  {GRID[-1]}")

候选频带 55 个，带宽 500 / 1000 / 2000 Hz，中心每 250 Hz 一档
例如: (250, 750)  (2500, 3000)  (5250, 5750)


### 一个藏在 `src/envelope.py` 里的细节，值得单独说

模块里有这么一段：

```python
EDGE_FRACTION = 0.05   # 掐掉首尾各 5%
```

**这是我调试时被咬了一口才加上的。**

`filtfilt` 在信号两端会"振铃" —— 滤波器启动和收尾时产生的人为暂态。窄带滤波器振得尤其久。

这些暂态**又尖又突兀**，所以：

- 它们会把**峭度**顶得极高（峭度专门对尖刺敏感，还记得步骤 02 吧）
- 于是任何"按峭度选带"的方法，都可能选中一个**只是边缘在振铃**的频带

我第一次跑这一节时，按峭度给外圈选出的是 **250–750 Hz** —— 一个毫无道理的低频带。掐掉首尾 5% 重跑，它变成了 4750–5750 Hz。

**换句话说，之前那个选择完全是滤波器自己造出来的假象，和轴承无关。**

> 这类 bug 最坏的地方在于它不报错。程序跑得好好的，结果是错的，而且错得"看起来像个结果"。**只有当你拿它和别的东西交叉验证时才会露馅。**

---
## 4. 图 6a：频带地图

对每条记录，把 55 个候选频带都评一遍分，画成一张图：横轴是频带中心，纵轴是带宽，颜色是得分。

**这张图叫"频带地图"** —— 它把"哪一段频率适合做包络分析"这件事直接画出来了。

> 这个思路在文献里有个近亲叫 **kurtogram**（谱峭度图，Antoni 2006），做法是用峭度当颜色。我们下面会看到峭度在这份数据上不好使，所以用了谐波梳评分。

这一格要跑十几秒（55 个频带 × 4 条记录 × 每次滤波+希尔伯特+FFT）。

In [ ]:
def scan(sig):
    ff = sig.fault_freqs()
    rows = []
    for band in GRID:
        score, which, _ = ev.score_band(sig.x[:N], band, ff)
        xf = ev.band_pass(sig.x[:N], band)
        f, a = ev.envelope_spectrum(sig.x[:N], band)
        truth = TRUTH[sig.name]
        rows.append(dict(band=band, comb=score, which=which,
                         kurt=kurtosis(xf, fisher=False),
                         ratio=ev.peak_ratio(f, a, ff[truth]) if truth else np.nan))
    return rows

scans = {}
for s in signals:
    scans[s.name] = scan(s)
    best = max(scans[s.name], key=lambda r: r["comb"])
    print(f"{s.name:<13} 最佳 {str(best['band']):>14}  comb={best['comb']:7.2f}  "
          f"判定 {best['which']}")

In [ ]:
WIDTHS = sorted({b[1] - b[0] for b in GRID})

fig6a, axes = plt.subplots(2, 2, figsize=(12.5, 8), layout="constrained")

vmax = max(r["comb"] for rows in scans.values() for r in rows)

for ax, s in zip(axes.ravel(), signals):
    rows = scans[s.name]
    # 摆成 带宽 x 中心 的网格
    centres = sorted({(b[0] + b[1]) / 2 for b in GRID})
    M = np.full((len(WIDTHS), len(centres)), np.nan)
    for r in rows:
        lo, hi = r["band"]
        M[WIDTHS.index(hi - lo), centres.index((lo + hi) / 2)] = r["comb"]

    im = ax.pcolormesh(centres, range(len(WIDTHS)), M,
                       cmap="viridis", shading="nearest",
                       norm=plt.matplotlib.colors.LogNorm(vmin=1, vmax=vmax))
    ax.set_yticks(range(len(WIDTHS)))
    ax.set_yticklabels([f"{w}" for w in WIDTHS])
    ax.set_ylabel("Bandwidth (Hz)")
    ax.set_xlabel("Band centre (Hz)")

    best = max(rows, key=lambda r: r["comb"])
    bc = (best["band"][0] + best["band"][1]) / 2
    ax.plot(bc, WIDTHS.index(best["band"][1] - best["band"][0]),
            marker="o", markersize=11, markerfacecolor="none",
            markeredgecolor="w", markeredgewidth=1.8)
    ax.set_title(f"{s.name}   best {best['band']}  "
                 f"comb {best['comb']:.1f}  -> {best['which']}", fontsize=10)

fig6a.colorbar(im, ax=axes, label="comb score (log)", pad=0.015, aspect=35)
fig6a.suptitle("Fig. 6a  Band map: harmonic-comb score over candidate bands\n"
               "circle marks the blind pick; score is blind to which fault is present",
               fontsize=11)
plt.show()

### 读图 6a

**Outer race**：2500–4000 Hz 中心那一带亮成一片，而且**范围很宽** —— 说明这个选择不挑剔，稍微偏一点也照样成立。**结论稳健。**

**Inner race**：亮区在 1000–2500 Hz，位置和外圈**明显不同**。

**Healthy**：整张图都是暗的。没有任何频带能让健康记录冒出高分 —— 这正是应该的。

**Ball**：也基本是暗的，只有零星几个中等格子。**方法在告诉你：这条记录我给不出可靠答案。**

> 注意看外圈那张的形状：亮区是**连成一片**的，不是孤立的亮点。这很重要 —— **如果最优解是一个孤立的点，说明它多半是噪声撞出来的；连成片才说明背后有真实的物理结构。**
>
> 这条经验适用于所有参数扫描：**看最优点周围长什么样，比看最优值本身更能判断结论可不可信。**

---
## 5. 为什么不用峭度

上面提过 kurtogram 用峭度当打分。这里把两种准则放在一起比。

In [ ]:
print("三种选带准则，各自选出的频带，以及它实际拿到的诊断质量\n")
print("  A 带内峭度        (盲)")
print("  B 谐波梳评分      (盲)")
print("  C 自己故障频率峰背比 (非盲 —— 作弊，只作为上限参考)\n")

for crit, label in [("kurt", "A 峭度"), ("comb", "B 谐波梳"), ("ratio", "C 作弊上限")]:
    print(f"=== 按 {label} 选带 ===")
    print(f"{'记录':<13}{'选中频带':<15}{'实得峰背比':>12}{'可达最好':>10}{'保留':>9}")
    print("-" * 62)
    for s in signals:
        rows = scans[s.name]
        if crit == "ratio" and TRUTH[s.name] is None:
            continue
        valid = [r for r in rows if not (crit == "ratio" and np.isnan(r["ratio"]))]
        pick = max(valid, key=lambda r: r[crit])
        top = np.nanmax([r["ratio"] for r in rows])
        got = pick["ratio"]
        pct = "-" if np.isnan(got) or np.isnan(top) else f"{100*got/top:.1f}%"
        gs = "-" if np.isnan(got) else f"{got:.1f}"
        ts = "-" if np.isnan(top) else f"{top:.1f}"
        print(f"{s.name:<13}{str(pick['band']):<15}{gs:>12}{ts:>10}{pct:>9}")
    print()

### 峭度在这份数据上不好使

| 记录 | 峭度选中 | 实得 | 可达最好 | 保留 |
|---|---|---|---|---|
| Outer race | 4750–5750 | 141 | 588 | **24%** |
| Inner race | 2250–4250 | 204 | 317 | 64% |
| Ball | 5250–5750 | 3.1 | 11.6 | 27% |

**谐波梳选中的频带能拿到 95–99% 的可达上限，峭度只有 24–64%。**

**为什么峭度会失手？**

峭度回答的是"这段信号里有没有突兀的尖刺"。但**尖刺的来源不止一种**：

- 轴承撞击 ✓ 我们要的
- 电气噪声、传感器打火 ✗
- 滤波器边缘振铃 ✗（就是上面那个坑）
- 单次偶然的机械碰撞 ✗

**峭度分不清它们。** 它只知道"尖"。

谐波梳评分要求的是**周期性**：不但要有冲击，还得**按一个整数倍频率列排布**。一次偶然的碰撞、一段振铃，都做不到这一点。

> 所以这不是"峭度是个坏指标"，而是**它测的东西不够具体**。
>
> 这一条很值得带走：**选指标时，要问它测的到底是不是你要的那个东西。** 峭度测"尖"，我们要"周期性的尖" —— 差这一个词，结果差 4 倍。

In [ ]:
# 图 6b：两种准则在同一条记录上的走向（外圈，带宽 1000 Hz）
rows = [r for r in scans["Outer race"] if r["band"][1] - r["band"][0] == 1000]
centres = [(r["band"][0] + r["band"][1]) / 2 for r in rows]

fig6b, ax1 = plt.subplots(figsize=(11, 4.2))
ax2 = ax1.twinx()

l1, = ax1.plot(centres, [r["comb"] for r in rows], "o-", color="#1B4F8F",
               markersize=4, label="comb score (used)")
l2, = ax2.plot(centres, [r["kurt"] for r in rows], "s--", color="#9A5F0A",
               markersize=4, alpha=0.8, label="band kurtosis (rejected)")
l3, = ax1.plot(centres, [r["ratio"] for r in rows], "^:", color="#8E3320",
               markersize=4, alpha=0.7, label="true BPFO ratio (cheating)")

ax1.set_yscale("log")
ax1.set_xlabel("Band centre (Hz),  bandwidth 1000 Hz")
ax1.set_ylabel("comb score / peak ratio (log)")
ax2.set_ylabel("kurtosis")
ax1.grid(alpha=0.25)
ax1.legend(handles=[l1, l3, l2], loc="upper left", fontsize=9)
ax1.set_title("Fig. 6b  Outer race: the comb score tracks the true quality, "
              "kurtosis does not", fontsize=11)
fig6b.tight_layout()
plt.show()

**蓝线（谐波梳）和红线（作弊上限）走势几乎重合** —— 谐波梳在盲的情况下，准确地跟上了"这个频带到底好不好"。

**橙色虚线（峭度）走的是另一条路**，峰值在完全不同的地方。

这张图就是"为什么选谐波梳"的全部论据。**报告里放这一张，比写三段话管用。**

---
## 6. 阈值定在哪 —— 用健康记录标定

现在有了选带方法，还差最后一块：**得分多高才算"确实有故障"？**

不能拍脑袋定。**用健康记录标定。**

关键在于：健康记录必须**走完全一样的流程**，包括在同样 55 个频带里搜索、取最高分。

**为什么必须这样？** 因为"在 55 个里挑最高的"这个动作本身就会抬高分数 —— 一堆带噪声的数里取最大值，天然比其中任何一个大。这叫**选择偏倚**（selection bias）。

如果你用"单个频带的典型分数"当阈值，就低估了这个偏倚，会造出假阳性。

In [ ]:
hc = np.array([r["comb"] for r in scans["Healthy"]])

print("健康记录在 55 个频带上的 comb 得分分布")
print(f"  中位数  {np.median(hc):6.2f}")
print(f"  90 分位 {np.percentile(hc, 90):6.2f}")
print(f"  最大值  {hc.max():6.2f}   <- 搜索 55 次后能刷到的最高分")
print()
print("如果只看单个频带，中位数 3.6 会让你以为阈值定 5 就够了。")
print(f"实际上搜索之后健康记录能到 {hc.max():.1f} —— 阈值必须定在这之上。\n")

THRESHOLD = hc.max()
print(f"决策规则:  comb > {THRESHOLD:.2f}  才下诊断结论，否则报告'无可靠结论'\n")
print("=" * 66)
print(f"{'记录':<13}{'最佳频带':<15}{'comb':>8}{'/阈值':>8}   {'判定':<8}{'真值':<8}")
print("-" * 66)
for s in signals:
    best = max(scans[s.name], key=lambda r: r["comb"])
    call = best["which"] if best["comb"] > THRESHOLD else "无结论"
    truth = TRUTH[s.name] or "健康"
    ok = "OK" if (call == truth) or (truth == "健康" and call == "无结论") \
        or (truth == "BSF" and call == "无结论") else "x"
    print(f"{s.name:<13}{str(best['band']):<15}{best['comb']:>8.2f}"
          f"{best['comb']/THRESHOLD:>8.1f}x   {call:<8}{truth:<8}{ok}")

---
## 7. 结果 —— 一条可辩护的规则

**完整的流程，全程不使用"答案"：**

1. 从轴承几何和转速算出三个候选频率 BPFO / BPFI / BSF
2. 在 55 个候选频带上，各算一次包络谱
3. 每个频带对三个候选频率各算谐波梳评分，取最大值作为该频带得分
4. 选得分最高的频带；它支持的那个候选频率就是诊断结论
5. **健康记录走同样的流程，它的最高分作为阈值**
6. 超过阈值才下结论，否则报"无可靠结论"

**结果：**

| 记录 | 选中频带 | comb | 相对阈值 | 判定 | 真值 |
|---|---|---|---|---|---|
| Healthy | 750–1250 | 11.06 | 1.0× | 无结论 | 健康 ✓ |
| Outer race | 2750–3750 | 431.9 | **39×** | BPFO | BPFO ✓ |
| Inner race | 1250–2250 | 226.9 | **21×** | BPFI | BPFI ✓ |
| Ball | 2500–3000 | 9.66 | 0.9× | **无结论** | BSF ✓（拒答正确） |

### 滚珠那一行是这套方法最好的部分

它的得分 9.66 **低于健康记录能刷到的 11.06**。所以规则说：**不下结论。**

**这不是失败，这是正确行为。** 一个诊断系统最危险的不是漏诊，是**在没有证据时自信地给出答案**。

如果去掉阈值、硬取最高分，它会报"BPFO"——**一个错误的诊断**。阈值把它拦住了。

> **但先别急着庆祝。** 这只是 0.007″ 那一条。第 8 节会把 0.014″ 和 0.021″ 也拿来测，**其中一条会越过阈值并给出错误答案**。
>
> 在只有四条记录时下的结论，很可能在第五条上翻车 —— 这也是为什么必须做第 8 节那个稳健性测试。

---
## 8. 稳健性：换个坑的大小还成立吗

到这儿我们只验证了 0.007 英寸那一组。**一个只在单一条件下成立的方法没有价值。**

我们手上有 0.014 和 0.021 英寸的记录（步骤 02 之后下载的）。拿来做两个测试：

**测试 1**：用 0.007″ 上选出的频带，直接套到 0.014″ 和 0.021″ 上，还管用吗？
**测试 2**：每条记录各自重新盲选频带，诊断还对吗？

In [ ]:
SEVERITY = [
    ("Outer", "0.007", "OR007at6_1hp_131.mat", "BPFO"),
    ("Outer", "0.014", "OR014at6_1_198.mat",   "BPFO"),
    ("Outer", "0.021", "OR021at6_1_235.mat",   "BPFO"),
    ("Inner", "0.007", "IR007_1hp_106.mat",    "BPFI"),
    ("Inner", "0.014", "IR014_1_170.mat",      "BPFI"),
    ("Inner", "0.021", "IR021_1_210.mat",      "BPFI"),
    ("Ball",  "0.007", "B007_1hp_119.mat",     "BSF"),
    ("Ball",  "0.014", "B014_1_186.mat",       "BSF"),
    ("Ball",  "0.021", "B021_1_223.mat",       "BSF"),
]
FIXED = {"Outer": (2750, 3750), "Inner": (1250, 2250), "Ball": (2500, 3000)}

print("固定频带 = 在 0.007in 记录上盲选出来的那个")
print(f"comb 是决策变量，阈值 {THRESHOLD:.2f}；峰背比只是参考\n")
print(f"{'故障':<7}{'坑':<7}{'固定带':>9}{'重新盲选的带':>15}{'重选峰背比':>11}"
      f"{'comb':>9}{'/阈值':>7}{'判定':>6}   结果")
print("-" * 82)
sev_rows = []
for loc, size, fname, key in SEVERITY:
    s = cwru_io.load(DATA / fname)
    ff = s.fault_freqs()
    f, a = ev.envelope_spectrum(s.x[:N], FIXED[loc])
    fixed = ev.peak_ratio(f, a, ff[key])
    band, score, which, _ = ev.select_band(s.x[:N], ff, GRID)
    f2, a2 = ev.envelope_spectrum(s.x[:N], band)
    got = ev.peak_ratio(f2, a2, ff[key])
    if score <= THRESHOLD:
        verdict, ok = "拒答", "declined"
    elif which == key:
        verdict, ok = "正确", "correct"
    else:
        verdict, ok = "** 误诊 **", "false"
    sev_rows.append((loc, size, fixed, band, got, score, which, ok))
    print(f"{loc:<7}{size:<7}{fixed:>9.1f}{str(band):>15}{got:>11.1f}"
          f"{score:>9.2f}{score/THRESHOLD:>6.1f}x{which:>6}   {verdict}")

### 这张表说了四件事，其中一件是坏消息

**① 最佳共振带不只随故障位置变，还随坑的大小变。**

外圈 0.007″ 的最佳带是 2750–3750，0.021″ 变成了 1250–2250。**同一个位置、同一台机器，坑大一点，最适合的频带就搬家了。**

物理上说得通：坑越大，撞击的接触过程越长，激起的频率成分越偏低。

**② 固定频带会付出代价，有时很大。**

外圈 0.014″ 用固定带只有 6.9，重选到 14.1。这条记录本来就难 —— 步骤 02 里它的峭度是 2.94，比健康的 2.98 还低。

**③ 六条内外圈记录，重新盲选之后全部判对。**

这是最重要的一条：**最优频带不稳定，但诊断结论稳定。** 方法依赖的不是某个精心挑选的参数，而是真实的物理结构。

> 如果结论随参数剧烈变化，那是在调参；结论稳定、只有中间量在变，那才是在测量。**这个区别是判断一个方法可不可信的核心。**

**④ 坏消息：滚珠 0.021″ 产生了一次误诊。**

| | comb | /阈值 | 判定 | 真值 |
|---|---|---|---|---|
| Ball 0.007″ | 9.66 | 0.9× | 拒答 | BSF |
| Ball 0.014″ | 10.37 | 0.9× | 拒答 | BSF |
| Ball 0.021″ | **14.52** | **1.3×** | **BPFI** | BSF |

**它越过了阈值，然后报了个错误的答案。** 九条故障记录里一次假阳性。

**我不打算把阈值调高来掩盖它。** 因为看下一段就知道，调高会同时毁掉一个正确答案。

---
## 8b. 阈值附近是一片灰色地带 —— 这才是诚实的结论

把九条记录按"相对阈值的倍数"排开：

| 倍数 | 记录 | 结果 |
|---|---|---|
| 39.1× | Outer 0.007 | 正确 |
| 20.5× | Inner 0.007 | 正确 |
| 7.7× | Outer 0.021 | 正确 |
| 6.3× | Inner 0.021 | 正确 |
| 3.3× | Inner 0.014 | 正确 |
| **1.3×** | **Ball 0.021** | **误诊** |
| **1.2×** | **Outer 0.014** | **正确（勉强）** |
| 0.9× | Ball 0.014 | 拒答 |
| 0.9× | Ball 0.007 | 拒答 |

**唯一那次误诊（1.3×），和勉强答对的那条（1.2×），挤在一起。**

这说明单一阈值不够用 —— 把它提到 1.5× 能挡掉误诊，但同时会把外圈 0.014″ 这个**正确**结论也挡掉。**两者在这个指标上分不开。**

### 所以改成三档

| comb / 阈值 | 结论 | 本数据上的表现 |
|---|---|---|
| **> 5×** | 确诊 | 4 条，**全对** |
| 1× ~ 5× | 存疑，建议复测或换工况 | 3 条，2 对 1 错 |
| < 1× | 不下结论 | 2 条，都是滚珠 |

**所有错误都落在"存疑"档里。** 这是一个诊断系统能给出的最有用的性质 —— 不是"永不犯错"，而是**"犯错时自己知道没把握"**。

> 这一段是整个项目里最值得写进报告的东西。
>
> 一个只报成功率的结果（"6/9 正确"）说明不了什么。**一个能说出"我的错误都发生在哪个区间、为什么、以及怎么识别那个区间"的结果，才是工程结论。**

In [ ]:
# 图 6c：稳健性
fig6c, (axL, axR) = plt.subplots(1, 2, figsize=(12.5, 4.3))

locs = ["Outer", "Inner", "Ball"]
sizes = ["0.007", "0.014", "0.021"]
w = 0.26
CL = {"Outer": "#8E3320", "Inner": "#1B4F8F", "Ball": "#9A5F0A"}

for j, size in enumerate(sizes):
    vals = [next(r[5] for r in sev_rows if r[0] == loc and r[1] == size) for loc in locs]
    axL.bar(np.arange(3) + (j - 1) * w, vals, width=w,
            color=[CL[l] for l in locs], alpha=0.45 + 0.22 * j,
            label=f'{size}"')
axL.axhline(THRESHOLD, color="#5B6773", linestyle="--", linewidth=1.2)
axL.set_xlim(-0.55, 3.25)          # 右边留白，给阈值标注让位
axL.text(2.52, THRESHOLD * 1.05, f"threshold\n{THRESHOLD:.1f}\n(healthy,\nsame search)",
         fontsize=8, ha="left", va="bottom", color="#5B6773", linespacing=1.35)
axL.set_yscale("log")
axL.set_xticks(range(3))
axL.set_xticklabels(locs)
axL.set_ylabel("comb score (log)")
axL.set_title("Blind comb score after re-selecting the band", fontsize=10)
axL.legend(fontsize=8, title="fault size", title_fontsize=8)
axL.grid(axis="y", alpha=0.25)
axL.set_axisbelow(True)

for loc in locs:
    xs = [float(r[1]) for r in sev_rows if r[0] == loc]
    ys = [(r[3][0] + r[3][1]) / 2 for r in sev_rows if r[0] == loc]
    axR.plot(xs, ys, "o-", color=CL[loc], label=loc, markersize=7)
axR.set_xlabel("Fault diameter (in)")
axR.set_ylabel("Centre of the selected band (Hz)")
axR.set_xticks([0.007, 0.014, 0.021])
axR.set_title("The best band moves with fault size", fontsize=10)
axR.legend(fontsize=9)
axR.grid(alpha=0.25)

fig6c.suptitle("Fig. 6c  Robustness across fault size, all at 1 hp", fontsize=11)
fig6c.tight_layout()
plt.show()

---
## 9. 如果只能用一个固定频带

实际的在线监测系统往往不能每次都重新搜索（算力、实时性）。那么**一个统一的频带**最好选哪一段？

In [ ]:
CANDIDATES = [(2000, 4000), (2750, 3750), (1250, 2250), (1000, 3000), (2000, 3000)]
six = [r for r in SEVERITY if r[0] in ("Outer", "Inner")]

print("各候选统一频带，在六条外圈/内圈记录上的峰背比\n")
hdr = "".join(f"{loc[0]}{size[2:]:>4}" for loc, size, _, _ in six)
print(f"{'统一频带':<14}" + "".join(f"{loc[:2]+size[1:]:>12}" for loc, size, _, _ in six)
      + f"{'几何均值':>11}")
print("-" * 97)
for band in CANDIDATES:
    vals = []
    for loc, size, fname, key in six:
        s = cwru_io.load(DATA / fname)
        f, a = ev.envelope_spectrum(s.x[:N], band)
        vals.append(ev.peak_ratio(f, a, s.fault_freqs()[key]))
    gm = np.exp(np.mean(np.log(vals)))
    print(f"{str(band):<14}" + "".join(f"{v:>12.1f}" for v in vals) + f"{gm:>11.1f}")

print("\n用几何均值排名：它奖励'哪条都不太差'，惩罚'某条特别差'——")
print("这正是选统一参数时该要的性质。")

**结论：如果必须固定一个频带，选 1000–3000 Hz。**

它在六条记录上的几何均值最高 —— 不是在任何单条上最好，但**没有一条特别差**。

而我在步骤 05 里随手定的 2000–4000 Hz？它在外圈 0.014″ 上掉到 5.9。**手划的那一下确实不够好，只是恰好在最初那四条记录上没暴露出来。**

> 这就是这一节存在的意义。不做这一步，你永远不知道自己那个"看着差不多"的参数在别的条件下会不会塌。

---
## 10. 这一步在报告里怎么写

路线图里说过：招生官真正在看的就是这一步。因为前五步证明你会用工具，**这一步证明你会做判断**。

写的时候按这个结构，**每一条都要有图或数字支撑**：

| 段落 | 写什么 | 证据 |
|---|---|---|
| 问题 | 共振带的选择影响结果，而它没有标准答案 | 练习里的扫描表：外圈 682 / 内圈 373，最佳带不同 |
| 陷阱 | 用"答案"选参数是循环论证 | 明确区分"几何已知"和"故障未知" |
| 方法 | 谐波梳评分，盲于故障类型 | 公式 + 为什么用几何平均 |
| 为什么不用峭度 | 峭度测"尖"，我们要"周期性的尖" | **图 6b** —— 两条曲线走向不同 |
| 阈值 | 健康记录走同样流程的最高分 | 选择偏倚：单带中位数 3.6 vs 搜索后 11.1 |
| 结果 | 外圈 39×、内圈 21× 阈值 | **图 6a** 频带地图 |
| 稳健性 | 最优带随故障尺寸移动，但六条内外圈诊断不变 | **图 6c** |
| **失败案例** | 滚珠 0.021″ 越过阈值并误诊 | 九条里一次假阳性，**必须写** |
| **三档规则** | >5× 确诊、1–5× 存疑、<1× 不下结论 | 所有错误都在"存疑"档 |
| 实用建议 | 固定带取 1000–3000 Hz | 几何均值排名 |
| 局限 | 单一数据集、单一转速、实验台数据 | 老实写 |

**有三句话是这一节的核心，务必写进去：**

> 1. **"最优共振带随故障位置和故障尺寸移动，但诊断结论在全部六条内外圈记录上保持正确 —— 方法依赖的是物理结构，不是某个调好的参数。"**
>
> 2. **"判据阈值由健康记录在完全相同的搜索流程下的最高分标定，因此计入了搜索本身带来的选择偏倚（单频带中位数 3.6，搜索 55 个频带后 11.1）。"**
>
> 3. **"九条故障记录中出现一次假阳性（滚动体 0.021 英寸，判为内圈）。该样本与唯一一条勉强正确的样本（外圈 0.014 英寸）分别位于阈值的 1.3 倍与 1.2 倍处，二者在本判据下不可分。因此给出三档判定规则，全部错误均落在'存疑'档内。"**

**第 3 句是最重要的。** 它承认了一次失败，然后说明了失败发生在哪、为什么不能靠调阈值解决、以及怎么让系统在那种情况下自己举手。

> 一份只报成功的材料，懂行的人会怀疑你藏了东西或者根本没测够。**一份指出自己在哪失败、并且把失败圈起来的材料，才说明你真的在做工程。**

---
## 11. 保存

In [ ]:
FIGURES.mkdir(exist_ok=True)
for fig, name in [(fig6a, "fig06a_band_map.png"),
                  (fig6b, "fig06b_criterion_comparison.png"),
                  (fig6c, "fig06c_robustness.png")]:
    p = FIGURES / name
    fig.savefig(p, dpi=150, bbox_inches="tight")
    print("已保存:", p)

p = FIGURES / "table05_band_selection.txt"
with open(p, "w", encoding="utf-8") as fh:
    fh.write("Resonance band selected blind, and the diagnosis it supports\n")
    fh.write(f"Grid: {len(GRID)} bands, widths {sorted({b[1]-b[0] for b in GRID})} Hz, "
             f"centres every 250 Hz\n")
    fh.write(f"Threshold {THRESHOLD:.2f} = highest comb score the healthy record "
             f"reaches over the same search\n\n")
    fh.write(f"{'record':<14}{'band':<16}{'comb':>9}{'x thr':>8}  {'call':<8}{'truth':<8}\n")
    fh.write("-" * 66 + "\n")
    for s in signals:
        best = max(scans[s.name], key=lambda r: r["comb"])
        call = best["which"] if best["comb"] > THRESHOLD else "no call"
        fh.write(f"{s.name:<14}{str(best['band']):<16}{best['comb']:>9.2f}"
                 f"{best['comb']/THRESHOLD:>7.1f}x  {call:<8}"
                 f"{TRUTH[s.name] or 'healthy':<8}\n")
    fh.write("\nAcross fault size, 1 hp:\n")
    for loc, size, fixed, band, got, score, which, ok in sev_rows:
        fh.write(f"  {loc:<6}{size:<7}{str(band):<16}comb {score:>8.2f}  "
                 f"call {which:<6}{ok}\n")
print("已保存:", p)

---
## 12. 自己动手

每格独立，直接跑。

In [ ]:
# 练习 1：网格密一点，结论会变吗
# ------------------------------------------------
# 一个可信的结论不该对网格的疏密敏感
FINE = ev.band_grid(widths=(250, 500, 750, 1000, 1500, 2000), step=125)
print(f"细网格 {len(FINE)} 个频带（原来 {len(GRID)} 个），跑几十秒\n")

print(f"{'记录':<13}{'粗网格':<26}{'细网格':<26}")
print("-" * 66)
for s in signals:
    coarse = max(scans[s.name], key=lambda r: r["comb"])
    band, score, which, _ = ev.select_band(s.x[:N], s.fault_freqs(), FINE)
    left = f"{coarse['band']}  {coarse['comb']:.1f}  {coarse['which']}"
    right = f"{band}  {score:.1f}  {which}"
    print(f"{s.name:<13}{left:<26}{right:<26}")

print("\n判定结论变了吗？分数变了多少？阈值需要重新标定吗？")

In [ ]:
# 练习 2：谐波取几个最合适
# ------------------------------------------------
# 取太少 -> 单根峰也能得高分；取太多 -> 高次谐波本来就弱，拖累真信号
for nh in (1, 2, 3, 4, 6, 8):
    line = []
    for s in signals:
        ff = s.fault_freqs()
        f, a = ev.envelope_spectrum(s.x[:N], (2000, 4000))
        sc = max(ev.comb_score(f, a, ff[k], harmonics=nh) for k in ev.FAULT_KEYS)
        line.append(sc)
    ratio_to_healthy = [v / line[0] for v in line]
    print(f"  谐波数 {nh}:  " +
          "  ".join(f"{s.name[:5]}={v:7.1f}" for s, v in zip(signals, line)) +
          f"   |  故障/健康 = {ratio_to_healthy[1]:5.1f}x {ratio_to_healthy[2]:5.1f}x")

print("\n哪个谐波数让'故障 / 健康'的比值最大？那就是区分度最好的选择。")

In [ ]:
# 练习 3：录短一点还能选对带吗
# ------------------------------------------------
for secs in (1, 2, 5, 10):
    n = int(secs * FS)
    line = []
    for s in signals[1:]:
        band, score, which, _ = ev.select_band(s.x[:n], s.fault_freqs(), GRID)
        ok = "OK" if which == TRUTH[s.name] else "x"
        line.append(f"{s.name[:5]}: {str(band):<14}{score:6.1f} {which} {ok}")
    print(f"  {secs:2d} 秒   " + "   ".join(line))

print("\n录多久才够？短记录的分数为什么会变？（提示：频率分辨率）")

---
## 小结

| 做了 | 结论 |
|---|---|
| 识别循环论证 | 区分"几何已知的三个候选频率"和"未知的哪个坏" |
| 设计盲的打分 | 谐波梳（几何平均，要求每根谐波都在） |
| 频带地图（图 6a） | 外圈亮区连成一片 → 结论稳健，不是噪声撞出来的 |
| 对比峭度（图 6b） | 峭度只保留 24–64% 的可达质量；它测"尖"，我们要"周期性的尖" |
| 发现并修掉一个假象 | `filtfilt` 边缘振铃会骗过峭度，掐掉首尾 5% |
| 用健康记录标定阈值 | 单带中位数 3.6，搜索 55 次后 11.1 —— **选择偏倚必须计入** |
| 诊断结果（0.007″） | 外圈 **39×** 阈值、内圈 **21×**、滚珠 0.9× → 拒答 |
| 稳健性（图 6c） | 最优带随故障尺寸移动，**六条内外圈记录诊断全对** |
| **失败案例** | 滚珠 0.021″ 以 1.3× 越过阈值，**误诊为内圈** |
| **最终规则** | >5× 确诊（4 条全对）、1–5× 存疑（2 对 1 错）、<1× 不下结论 |
| 实用建议 | 固定带取 **1000–3000 Hz**（几何均值最高） |

**这一节和前五节的区别**：前面是"方法这样用"，这一节是"**为什么这样用，我怎么知道它靠得住，以及它什么时候不靠得住**"。

最后一句尤其重要。九条记录里那一次误诊不是这一节的瑕疵，**它是这一节最有价值的产出** —— 因为找到它、定位它、并说明为什么不能靠调阈值绕过去，正是"做判断"这件事本身。

**主线（步骤 01–06 + 报告）到此完成。** 剩下的步骤 07（分类器）是锦上添花 —— 路线图里明说了时间不够就砍它。